In [7]:
"""Code commo
Stratégie 3 Facteurs — Value + Momentum + Basis
Construction : terciles (long T3, short T1) par facteur
Portefeuille combiné = moyenne équipondérée des 3 portefeuilles facteur
(Backtest incluant les coûts de transaction réalistes)
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

LOCAL_FILE_PATH  = "data_finale.xlsx"
BCOM_FILE_PATH   = "bcom.xlsx"

VALUE_WIN_LONG   = 66
VALUE_WIN_SHORT  = 54
LOOKBACK_MOM     = 11
MIN_COMMOS       = 10
MONTHS_PER_YEAR  = 12
COMMOS_SELECTED  = None

# --- NOUVELLE HYPOTHÈSE DE COÛT DE TRANSACTION ---
DEFAULT_TC_BPS   = 20  # 20 points de base retenus par défaut pour le backtest


# ══════════════════════════════════════════════════════════════════════════════
# 1. CHARGEMENT DES DONNÉES
# ══════════════════════════════════════════════════════════════════════════════

def parse_sheet(rows):
    header_row = None
    for i, row in enumerate(rows):
        if row[0] == 'Date':
            header_row = i
            break
    if header_row is None:
        return pd.DataFrame()

    er_rec, f1_rec, f2_rec = [], [], []
    for row in rows[header_row + 1:]:
        try:
            na = lambda v: v is None or v == '#N/A N/A'
            if row[0]:
                er_rec.append({'date': pd.Timestamp(row[0]),
                               'ER'  : float(row[1]) if not na(row[1]) else np.nan})
            if len(row) > 4 and row[3]:
                f1_rec.append({'date': pd.Timestamp(row[3]),
                               'F1'  : float(row[4]) if not na(row[4]) else np.nan})
            if len(row) > 8 and row[7]:
                f2_rec.append({'date': pd.Timestamp(row[7]),
                               'F2'  : float(row[8]) if not na(row[8]) else np.nan})
        except (ValueError, TypeError):
            continue

    if not er_rec or not f1_rec or not f2_rec:
        return pd.DataFrame()

    df = (pd.DataFrame(er_rec).set_index('date')
          .join(pd.DataFrame(f1_rec).set_index('date'), how='outer')
          .join(pd.DataFrame(f2_rec).set_index('date'), how='outer')
          .sort_index().reset_index())
    return df


def load_data(path):
    from openpyxl import load_workbook
    wb     = load_workbook(path, read_only=True, data_only=True)
    sheets = COMMOS_SELECTED if COMMOS_SELECTED else wb.sheetnames

    er_frames, f1_frames, f2_frames = [], [], []
    print(f"  {'Commodite':<22} {'Obs.':>8}  {'Debut':>12}  {'Fin':>12}")
    print("  " + "─" * 60)

    for name in sheets:
        if name not in wb.sheetnames:
            print(f"  !  '{name}' introuvable, ignore.")
            continue
        rows = list(wb[name].iter_rows(values_only=True))
        df   = parse_sheet(rows)
        if df.empty or df['F1'].dropna().empty or df['F2'].dropna().empty:
            print(f"  x  {name:<22} donnees insuffisantes")
            continue

        er = df.dropna(subset=['ER']).set_index('date')['ER'].resample('ME').last()
        f1 = df.dropna(subset=['F1']).set_index('date')['F1'].resample('ME').last()
        f2 = df.dropna(subset=['F2']).set_index('date')['F2'].resample('ME').last()
        er.name = f1.name = f2.name = name
        er_frames.append(er)
        f1_frames.append(f1)
        f2_frames.append(f2)

        n = df.dropna(subset=['ER', 'F1', 'F2'], how='all').shape[0]
        print(f"  ok {name:<22} {n:>8}  "
              f"  {df['date'].min().date()!s:>12}  {df['date'].max().date()!s:>12}")

    wb.close()

    if not er_frames:
        raise RuntimeError(f"Aucune donnee chargee depuis '{path}'.")

    prices_er = pd.concat(er_frames, axis=1).sort_index()
    prices_f1 = pd.concat(f1_frames, axis=1).sort_index()
    prices_f2 = pd.concat(f2_frames, axis=1).sort_index()
    returns   = np.log(prices_er / prices_er.shift(1))
    return returns, prices_f1, prices_f2


def load_bcom(path):
    from openpyxl import load_workbook

    def excel_date(n):
        return datetime(1899, 12, 30) + timedelta(days=int(n))

    wb   = load_workbook(path, read_only=True, data_only=True)
    ws   = wb.active
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    header = next(i for i, r in enumerate(rows) if r[0] == 'Date')
    recs   = []
    for row in rows[header + 1:]:
        try:
            if not row[0] or not row[1]:
                continue
            date  = (excel_date(row[0]) if isinstance(row[0], (int, float))
                     else pd.Timestamp(row[0]))
            price = float(row[1])
            recs.append({'date': date, 'price': price})
        except (ValueError, TypeError):
            continue

    df      = pd.DataFrame(recs).sort_values('date').set_index('date')
    monthly = df['price'].resample('ME').last().dropna()
    bcom    = np.log(monthly / monthly.shift(1)).dropna()
    bcom.name = 'BCOM'
    return bcom


# ══════════════════════════════════════════════════════════════════════════════
# 2. SIGNAUX BRUTS
# ══════════════════════════════════════════════════════════════════════════════

def compute_value_signal(f1_prices):
    sum_long  = f1_prices.rolling(VALUE_WIN_LONG,  min_periods=VALUE_WIN_LONG).sum()
    sum_short = f1_prices.rolling(VALUE_WIN_SHORT, min_periods=VALUE_WIN_SHORT).sum()
    f1_mean   = (sum_long - sum_short) / (VALUE_WIN_LONG - VALUE_WIN_SHORT)
    return np.log(f1_mean / f1_prices)

def compute_mom_signal(returns, lookback=11):
    returns_skipped = returns.shift(1)
    return returns_skipped.rolling(lookback, min_periods=lookback).sum()

def compute_basis_signal(f1_prices, f2_prices, delta_T=1/12, smooth_window=12):
    carry_raw    = (f1_prices - f2_prices) / (f2_prices * delta_T)
    carry_smooth = carry_raw.rolling(window=smooth_window, min_periods=smooth_window).mean()
    return carry_smooth


# ══════════════════════════════════════════════════════════════════════════════
# 3. CONSTRUCTION DES PORTEFEUILLES PAR TERCILES
# ══════════════════════════════════════════════════════════════════════════════

def tercile_weights(signal_row):
    avail  = signal_row.dropna()
    n      = len(avail)
    if n < MIN_COMMOS:
        return pd.Series(dtype=float)

    t_size = n // 3
    if t_size == 0:
        return pd.Series(dtype=float)

    ranked = avail.rank(method='first', ascending=True)
    longs  = ranked[ranked > (n - t_size)].index
    shorts = ranked[ranked <= t_size].index

    w = pd.Series(0.0, index=avail.index)
    w[longs]  = +1.0 / len(longs)
    w[shorts] = -1.0 / len(shorts)
    return w

def make_weights_tercile(signal_df):
    rows = []
    for date, row in signal_df.iterrows():
        w = tercile_weights(row)
        if w.empty:
            continue
        w.name = date
        rows.append(w)
    if not rows:
        raise RuntimeError("Aucune date ne satisfait MIN_COMMOS pour les terciles.")
    return pd.DataFrame(rows).sort_index().shift(1)

def build_all_signals(returns, f1_prices, f2_prices):
    value_raw = compute_value_signal(f1_prices)
    mom_raw   = compute_mom_signal(returns, lookback=LOOKBACK_MOM)
    basis_raw = compute_basis_signal(f1_prices, f2_prices)

    w_value = make_weights_tercile(value_raw)
    w_mom   = make_weights_tercile(mom_raw)
    w_basis = make_weights_tercile(basis_raw)

    common_dates = (w_value.index.intersection(w_mom.index).intersection(w_basis.index))
    common_cols  = (w_value.columns.intersection(w_mom.columns).intersection(w_basis.columns))

    wv = w_value.loc[common_dates, common_cols].fillna(0)
    wm = w_mom.loc[common_dates, common_cols].fillna(0)
    wb = w_basis.loc[common_dates, common_cols].fillna(0)

    w_combined = (wv + wm + wb) / 3.0
    no_signal  = (wv == 0) & (wm == 0) & (wb == 0)
    w_combined[no_signal] = np.nan

    return (w_combined, w_value, w_mom, w_basis, value_raw, mom_raw, basis_raw)


# ══════════════════════════════════════════════════════════════════════════════
# 4. BACKTEST & COÛTS DE TRANSACTION
# ══════════════════════════════════════════════════════════════════════════════

def backtest_single(returns, weights, bcom=None, tc_bps=DEFAULT_TC_BPS):
    idx  = returns.index.intersection(weights.index)
    cols = returns.columns.intersection(weights.columns)
    r    = returns.loc[idx, cols]
    w    = weights.loc[idx, cols].fillna(0)

    # Calcul du Turnover : Somme absolue des changements de poids d'un mois sur l'autre
    w_prev = w.shift(1).fillna(0)
    turnover = (w - w_prev).abs().sum(axis=1)

    # Rendement Brut
    gross_ret = (w * r).sum(axis=1)

    # Coût de transaction évalué en fonction du turnover
    tc_penalty = turnover * (tc_bps / 10000.0)

    # Rendement Net
    strat_ret = gross_ret - tc_penalty
    n_active  = (w != 0).sum(axis=1)

    port = pd.DataFrame({
        'strat_return' : strat_ret,
        'turnover'     : turnover,
        'n_commos'     : n_active,
    }, index=idx).dropna(subset=['strat_return'])

    if bcom is not None:
        common = port.index.intersection(bcom.index)
        port   = port.loc[common]
        port['bh_return'] = bcom.loc[common]
    else:
        port['bh_return'] = r.loc[port.index].mean(axis=1)

    port['cum_strat'] = 100 * np.exp(port['strat_return'].cumsum())
    port['cum_bh']    = 100 * np.exp(port['bh_return'].cumsum())
    return port


# ══════════════════════════════════════════════════════════════════════════════
# 5. MÉTRIQUES ET AFFICHAGES
# ══════════════════════════════════════════════════════════════════════════════

def _ann_return(r):
    return np.exp(r.sum()) ** (MONTHS_PER_YEAR / len(r)) - 1

def _ann_vol(r):
    return r.std() * np.sqrt(MONTHS_PER_YEAR)

def _sharpe(r):
    v = _ann_vol(r)
    return _ann_return(r) / v if v > 0 else np.nan

def _max_drawdown(cum):
    return ((cum - cum.cummax()) / cum.cummax()).min()

def _calmar(r, cum):
    dd = abs(_max_drawdown(cum))
    return _ann_return(r) / dd if dd > 0 else np.nan

def _hit_rate(r):
    return (r > 0).mean()

def portfolio_metrics(port):
    r = port['strat_return'].dropna()
    turnover_ann = port['turnover'].mean() * MONTHS_PER_YEAR
    return {
        'Debut'                : port.index.min().strftime('%b %Y'),
        'Fin'                  : port.index.max().strftime('%b %Y'),
        'Nb mois'              : str(len(r)),
        'Turnover Ann. (X)'    : f"{turnover_ann:.1f}x",
        'Ret. annualise'       : f"{_ann_return(r):.2%}",
        'Volatilite'           : f"{_ann_vol(r):.2%}",
        'Sharpe'               : f"{_sharpe(r):.2f}",
        'Max Drawdown'         : f"{_max_drawdown(port['cum_strat']):.2%}",
        'Calmar'               : f"{_calmar(r, port['cum_strat']):.2f}",
        'Hit rate'             : f"{_hit_rate(r):.2%}",
    }

def print_tc_impact_table(returns, w_combined):
    print()
    inner_w = 48
    print('  ┌' + '─' * inner_w + '┐')
    print(f"  │ IMPACT DES COÛTS DE TRANSACTION (COMBINÉ)      │")
    print('  ├' + '─' * inner_w + '┤')
    print(f"  │ {'Frais (bps)':<13} │ {'Ret. Annualisé':<14} │ {'Sharpe':<12}│")
    print('  ├' + '─' * inner_w + '┤')

    for bps in range(0, 51, 10):
        port = backtest_single(returns, w_combined, tc_bps=bps)
        r_strat = port['strat_return'].dropna()
        if len(r_strat) < 2:
            ret_s, sh_s = '—', '—'
        else:
            ret_s = f"{_ann_return(r_strat):>+.2%}"
            sh_s  = f"{_sharpe(r_strat):>.2f}"

        # Surligner visuellement la ligne des 20 bps retenus
        if bps == DEFAULT_TC_BPS:
            print(f"  │ > {bps:>8} bp │ {ret_s:>14} │ {sh_s:>12} │  <-- RETENU")
        else:
            print(f"  │ {bps:>10} bp │ {ret_s:>14} │ {sh_s:>12} │")

    print('  └' + '─' * inner_w + '┘')

def print_comparison_table(pm_list, corr_matrix):
    labels_order = [
        'Debut', 'Fin', 'Nb mois', 'Turnover Ann. (X)',
        '__SEP__',
        'Ret. annualise', 'Volatilite', 'Sharpe',
        'Max Drawdown', 'Calmar', 'Hit rate',
    ]

    cols   = [lbl for lbl, _ in pm_list]
    W_lbl  = 22
    W_col  = 16
    n_cols = len(cols)
    box_w  = W_lbl + n_cols * W_col + 2

    print()
    print('╔' + '═' * box_w + '╗')
    title = f"RÉSULTATS NETS DE FRAIS ({DEFAULT_TC_BPS} bps de transaction)"
    print(f"║ {title:<{box_w - 1}}║")
    print('╠' + '═' * box_w + '╣')

    hdr = f"{'':<{W_lbl}}"
    for lbl in cols:
        hdr += f"{lbl:>{W_col}}"
    print(f"║ {hdr} ║")
    print('╟' + '─' * box_w + '╢')

    for k in labels_order:
        if k == '__SEP__':
            print('╟' + '─' * box_w + '╢')
            continue
        row = f"{k:<{W_lbl}}"
        for _, pm in pm_list:
            v = pm.get(k, '—')
            row += f"{v:>{W_col}}"
        print(f"║ {row} ║")

    print('╟' + '─' * box_w + '╢')
    strats = [lbl for lbl, _ in pm_list]
    for i in range(len(strats)):
        for j in range(i + 1, len(strats)):
            k   = f"Corr {strats[i]}/{strats[j]}"
            val = f"{corr_matrix.loc[strats[i], strats[j]]:.3f}"
            row = f"{k:<{W_lbl}}" + f"{val:>{W_col}}" + " " * (W_col * (n_cols - 1))
            print(f"║ {row} ║")

    print('╚' + '═' * box_w + '╝')

def print_positions(value_raw, mom_raw, basis_raw, w_value, w_mom, w_basis, w_combined):
    entries = [
        ('VALUE',    w_value,    value_raw),
        ('MOMENTUM', w_mom,      mom_raw),
        ('BASIS',    w_basis,    basis_raw),
        ('COMBINE',  w_combined, None),
    ]

    for label, w, sig in entries:
        last_date  = w.dropna(how='all').index[-2]
        next_month = (last_date + pd.offsets.MonthEnd(1)).strftime('%B %Y')
        w_row      = w.loc[last_date].dropna()
        w_row      = w_row[w_row != 0]

        pos_data = []
        for commo, wt in w_row.items():
            sv = np.nan
            if sig is not None and commo in sig.columns and last_date in sig.index:
                sv = sig.loc[last_date, commo]
            pos_data.append({'commo': commo, 'wt': wt, 'score': sv})

        if label == 'COMBINE':
            pos_data.sort(key=lambda x: x['wt'], reverse=True)
        else:
            pos_data.sort(key=lambda x: x['score'] if pd.notna(x['score']) else -9999, reverse=True)

        print()
        print(f"  ┌─ POSITIONS {label} (signal {last_date.strftime('%b %Y')} -> appliqué {next_month})")
        print(f"  │ {'Rg':>3}  {'Commodité':<22}  {'Poids':>9}  {'Score':>10}  {'Position':>9}")
        print(f"  │ {'─'*3}  {'─'*22}  {'─'*9}  {'─'*10}  {'─'*9}")

        for i, item in enumerate(pos_data, 1):
            commo, wt, sv = item['commo'], item['wt'], item['score']
            sv_s = f"{sv:>10.4f}" if pd.notna(sv) else f"{'—':>10}"
            tag = 'LONG' if wt > 1e-9 else 'SHORT' if wt < -1e-9 else '—'
            print(f"  │ {i:>3}  {commo:<22}  {wt:>9.4f}  {sv_s}  {tag:>9}")

        print(f"  └{'─'*61}")

FIXED_PERIODS = [
    ('2020-Present', 2020, None),
    ('2015-2019',    2015, 2019),
    ('2010-2014',    2010, 2014),
    ('2008-2009',    2008, 2009),
    ('2000-2007',    2000, 2007),
    ('Inception',    None, None),
]

def print_period_returns(ports, bcom):
    idx_common = ports[0][1].index
    for _, p in ports[1:]:
        idx_common = idx_common.intersection(p.index)
    idx_common = idx_common.intersection(bcom.index)

    end_date = idx_common.max()

    def get_mask(y_start, y_end):
        if y_start is None:
            return (idx_common >= idx_common.min()) & (idx_common <= end_date)
        t0 = pd.Timestamp(y_start, 1, 1) - pd.offsets.MonthEnd(1)
        t1 = pd.Timestamp(y_end, 12, 31) if y_end else end_date
        return (idx_common > t0) & (idx_common <= t1)

    def metrics_period(r_series):
        n = len(r_series)
        if n < 2: return '—', '—'
        ann_r = np.exp(r_series.sum()) ** (MONTHS_PER_YEAR / n) - 1
        ann_v = r_series.std() * np.sqrt(MONTHS_PER_YEAR)
        sh    = ann_r / ann_v if ann_v > 0 else np.nan
        return f"{ann_r:>+.2%}", f"{sh:>.2f}"

    all_labels = [lbl for lbl, _ in ports] + ['BCOM']
    W_lbl, W_ret, W_sh = 16, 9, 6
    pair_w = W_ret + W_sh + 2

    line_hdr = f"  │ {'Période':<{W_lbl}}"
    for col in all_labels:
        line_hdr += f" │ {col:^{pair_w}}"
    line_hdr += ' │'

    line_sub = f"  │ {'':<{W_lbl}}"
    for _ in all_labels:
        line_sub += f" │ {'Ret.':>{W_ret}}  {'Sh.':>{W_sh}}"
    line_sub += ' │'

    sep_inner = W_lbl + len(all_labels) * (pair_w + 3) - 1

    print()
    print('  ┌' + '─' * (sep_inner + 2) + '┐')
    print(f"  │ RENTABILITÉS NETTES PAR PÉRIODES ({DEFAULT_TC_BPS} bps){'':<{sep_inner - 42}} │")
    print('  ├' + '─' * (sep_inner + 2) + '┤')
    print(line_hdr)
    print(line_sub)
    print('  ├' + '─' * (sep_inner + 2) + '┤')

    for (period_label, y_start, y_end) in FIXED_PERIODS:
        mask = get_mask(y_start, y_end)
        sub_idx = idx_common[mask]

        row = f"  │ {period_label:<{W_lbl}}"
        for lbl, port in ports:
            r_sub = port.loc[sub_idx, 'strat_return'] if len(sub_idx) > 0 else pd.Series(dtype=float)
            ret_s, sh_s = metrics_period(r_sub)
            row += f" │ {ret_s:>{W_ret}}  {sh_s:>{W_sh}}"

        bcom_sub = bcom.loc[sub_idx] if len(sub_idx) > 0 else pd.Series(dtype=float)
        ret_b, sh_b = metrics_period(bcom_sub)
        row += f" │ {ret_b:>{W_ret}}  {sh_b:>{W_sh}} │"
        print(row)

    print('  └' + '─' * (sep_inner + 2) + '┘')

# ══════════════════════════════════════════════════════════════════════════════
# 7. GRAPHIQUES
# ══════════════════════════════════════════════════════════════════════════════

def plot_results(ports, save_path=None):
    fig = plt.figure(figsize=(14, 10))
    gs  = fig.add_gridspec(3, 1, height_ratios=[3, 1.5, 1.5], hspace=0.42)
    axes = [fig.add_subplot(gs[i]) for i in range(3)]
    ax1, ax2, ax3 = axes

    fig.suptitle(
        f"3 Facteurs : Value + Momentum + Basis (Net de Frais : {DEFAULT_TC_BPS} bps)\n"
        "Terciles Long T3 / Short T1 — Combine = moyenne equiponderee",
        fontsize=11, fontweight='bold'
    )

    idx_common = ports[0][1].index
    for _, p in ports[1:]:
        idx_common = idx_common.intersection(p.index)
    dates = idx_common.to_pydatetime()

    colors = ['#1F4E79', '#2E75B6', '#70AD47', '#ED7D31']
    styles = ['-', '-.', ':', '--']

    for (lbl, port), col, sty in zip(ports, colors, styles):
        r   = port.loc[idx_common, 'strat_return']
        cum = 100 * np.exp(r.cumsum())
        lw  = 2.2 if lbl == '3 Facteurs' else 1.3
        ax1.plot(dates, cum.values, color=col, lw=lw, linestyle=sty, label=lbl)

    cum_bcom = 100 * np.exp(ports[0][1].loc[idx_common, 'bh_return'].cumsum())
    ax1.plot(dates, cum_bcom.values, color='#C00000', lw=1.2, linestyle='--', alpha=0.8, label='BCOM')
    ax1.axhline(100, color='#aaa', lw=0.6, linestyle=':')
    ax1.set_ylabel('Valeur (base 100)', fontsize=10)
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_title('Rendements Cumules', fontsize=10, fontweight='bold')

    for (lbl, port), col, sty in zip(ports, colors, styles):
        r   = port.loc[idx_common, 'strat_return']
        cum = 100 * np.exp(r.cumsum())
        dd  = (cum - cum.cummax()) / cum.cummax() * 100
        lw  = 1.5 if lbl == '3 Facteurs' else 0.9
        if lbl == '3 Facteurs':
            ax2.fill_between(dates, dd.values, 0, color=col, alpha=0.2)
        ax2.plot(dates, dd.values, color=col, lw=lw, linestyle=sty, label=lbl)

    cum_bcom_dd = 100 * np.exp(ports[0][1].loc[idx_common, 'bh_return'].cumsum())
    dd_bcom = (cum_bcom_dd - cum_bcom_dd.cummax()) / cum_bcom_dd.cummax() * 100
    ax2.plot(dates, dd_bcom.values, color='#C00000', lw=1.0, linestyle='--', alpha=0.6, label='BCOM')
    ax2.axhline(0, color='#555', lw=0.6)
    ax2.set_ylabel('Drawdown (%)', fontsize=10)
    ax2.legend(fontsize=8, loc='lower left')
    ax2.grid(True, alpha=0.3)
    ax2.set_title('Drawdowns Compares', fontsize=10, fontweight='bold')

    r3 = ports[0][1].loc[idx_common, 'strat_return']
    colors_bar = ['#1F4E79' if x >= 0 else '#C00000' for x in r3]
    ax3.bar(dates, r3.values * 100, color=colors_bar, alpha=0.75, width=20)
    ax3.axhline(0, color='#555', lw=0.7)
    ax3.set_ylabel('Ret. mensuel (%)', fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_title('Rendements Mensuels — 3 Facteurs (Combine)', fontsize=10, fontweight='bold')
    ax3.set_xlabel('Date', fontsize=10)

    n_years = (idx_common.max() - idx_common.min()).days / 365
    locator = mdates.YearLocator(5 if n_years > 30 else 2 if n_years > 10 else 1)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax3.xaxis.set_major_locator(locator)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n  Graphique sauvegarde : {save_path}")
    plt.close(fig)

# ══════════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    print("\n" + "═" * 70)
    print("  STRATEGIE 3 FACTEURS — VALUE + MOMENTUM + BASIS")
    print(f"  (Incluant {DEFAULT_TC_BPS} bps de frais de transaction par defaut)")
    print("═" * 70)

    print(f"\n[1/4] Chargement des donnees...")
    returns, f1_prices, f2_prices = load_data(LOCAL_FILE_PATH)
    bcom = load_bcom(BCOM_FILE_PATH)

    print("\n[2/4] Construction des signaux et backtests...")
    (w_combined, w_value, w_mom, w_basis,
     value_raw, mom_raw, basis_raw) = build_all_signals(returns, f1_prices, f2_prices)

    # Affichage de l'impact des frais avant de retenir la version finale
    print_tc_impact_table(returns, w_combined)

    # Backtests officiels avec l'hypothèse de 20 bps
    port_combined = backtest_single(returns, w_combined, bcom)
    port_value    = backtest_single(returns, w_value,    bcom)
    port_mom      = backtest_single(returns, w_mom,      bcom)
    port_basis    = backtest_single(returns, w_basis,    bcom)

    ports = [
        ('3 Facteurs', port_combined),
        ('Value',      port_value),
        ('Momentum',   port_mom),
        ('Basis',      port_basis),
    ]

    idx_c = ports[0][1].index
    for _, p in ports[1:]:
        idx_c = idx_c.intersection(p.index)
    ret_df = pd.DataFrame({lbl: p.loc[idx_c, 'strat_return'] for lbl, p in ports})
    corr_matrix = ret_df.corr()

    print("\n[3/4] Metriques de performance")
    pm_list = [(lbl, portfolio_metrics(port)) for lbl, port in ports]
    print_comparison_table(pm_list, corr_matrix)
    print_positions(value_raw, mom_raw, basis_raw, w_value, w_mom, w_basis, w_combined)
    print_period_returns(ports, bcom)

    print("\n[4/4] Generation du graphique...")
    os.makedirs("charts", exist_ok=True)
    save_path = "charts/value_mom_basis_portfolio_net.png"
    plot_results(ports, save_path=save_path)

    print("\n  FIN D'EXECUTION\n" + "═" * 70)

    return returns, f1_prices, f2_prices, ports, w_combined, value_raw, mom_raw, basis_raw

if __name__ == "__main__":
    # Déstructurer explicitement le retour de main()
    returns, f1_prices, f2_prices, ports, w_combined, value_raw, mom_raw, basis_raw = main()

# Export Excel
output_path = "Commo_Strategy_Monthly_Returns.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for lbl, port in ports:
        sheet = lbl.replace(" ", "_").replace("é", "e")
        port[["strat_return"]].rename(
            columns={"strat_return": "Monthly_Return"}
        ).to_excel(writer, sheet_name=sheet)

print(f"Fichier exporté : {output_path}")

Exception ignored in: <function ZipFile.__del__ at 0x790199133ce0>
Traceback (most recent call last):
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1966, in __del__
    self.close()
  File "/usr/lib/python3.12/zipfile/__init__.py", line 1983, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file



══════════════════════════════════════════════════════════════════════
  STRATEGIE 3 FACTEURS — VALUE + MOMENTUM + BASIS
  (Incluant 20 bps de frais de transaction par defaut)
══════════════════════════════════════════════════════════════════════

[1/4] Chargement des donnees...
  Commodite                  Obs.         Debut           Fin
  ────────────────────────────────────────────────────────────
  ok WTI crude                 10621      1984-01-13    2026-03-20
  ok Brent crude                9717      1988-06-23    2026-03-20
  ok Natural gas                9052      1990-04-03    2026-03-20
  ok Heating oil                9998      1986-07-01    2026-03-20
  ok Gas oil                    9456      1989-07-03    2026-03-20
  ok RBOB                       9843      1987-01-14    2026-03-20
  ok Aluminium                  9017      1991-01-02    2026-03-20
  ok Copper                     9386      1988-12-06    2026-03-20
  ok Nickel                     9017      1991-01-02    20